# Notebook E: Q&A Instruction Fine-Tuning

**Goal**: Fine-tune a QLoRA adapter on top of the Phase 1 merged base model using the Unsloth framework and TRL `SFTTrainer` on Kaggle (T4 GPU).

## 0. Install

In [1]:

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl" peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 17.6 MB/s e

## 1. Environment Detection & Device Setup

In [2]:
import os
import sys

# Force single GPU visibility to prevent DDP multi-GPU configuration crashes on Kaggle T4 x2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Set CUDA_VISIBLE_DEVICES to '0'")

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    MODEL_NAME = "/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf"
    TRAIN_DS_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_train"
    VAL_DS_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_val"
    OUTPUT_DIR = "/kaggle/working/qa_checkpoints"
    FINAL_ADAPTER_DIR = "/kaggle/working/spurgeon_lora_qa"
elif IS_COLAB:
    MODEL_NAME = "/content/spurgeon_phase1_merged"
    TRAIN_DS_PATH = "/content/qa_dataset_train"
    VAL_DS_PATH = "/content/qa_dataset_val"
    OUTPUT_DIR = "/content/qa_checkpoints"
    FINAL_ADAPTER_DIR = "/content/spurgeon_lora_qa"
else:
    MODEL_NAME = "../models/spurgeon_phase1_merged"
    TRAIN_DS_PATH = "../data/qa_dataset_train"
    VAL_DS_PATH = "../data/qa_dataset_val"
    OUTPUT_DIR = "../models/qa_checkpoints"
    FINAL_ADAPTER_DIR = "../models/spurgeon_lora_qa"

print(f"Base Model: {MODEL_NAME}")
print(f"Train dataset: {TRAIN_DS_PATH}")

Set CUDA_VISIBLE_DEVICES to '0'
Base Model: /kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf
Train dataset: /kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_train


## 2. Load Model & Setup LoRA Adapter

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import PreTrainedTokenizerFast
from unsloth.chat_templates import get_chat_template

# Monkey-patch the incorrect Hugging Face mistral regex warning/patching bug for Qwen model
PreTrainedTokenizerFast._patch_mistral_regex = classmethod(lambda cls, tokenizer, *args, **kwargs: tokenizer)

MAX_SEQ_LENGTH = 2048
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Apply the correct ChatML template for Qwen2.5
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# ============================================================================
# CRITICAL FIX: make <|im_end|> an ATOMIC special token + the EOS.
#
# The Phase-1 base comes from a GGUF round-trip whose special-token table is
# broken: eos_token prints as '' (id 151643 / <|endoftext|>) and, worse, the
# turn terminator "<|im_end|>" can tokenize into ORDINARY subwords instead of
# the single id 151645. When that happens the SFT training text "...<|im_end|>"
# teaches the model to "end" a turn by emitting junk subwords (which surface as
# "vinfos" / "spepacer" at inference) that NO eos_token_id can ever stop on.
#
# We force <|im_end|> / <|im_start|> to be registered special tokens, set EOS to
# <|im_end|>, align the model configs, and then ASSERT the terminator encodes to
# exactly one id. If the assert fails, the dataset must be re-tokenized with a
# clean tokenizer before training (do NOT proceed -- training would be wasted).
# ============================================================================
from transformers import AddedToken

tokenizer.add_special_tokens(
    {
        "eos_token": "<|im_end|>",
        "additional_special_tokens": [
            AddedToken("<|im_start|>", special=True, normalized=False),
            AddedToken("<|im_end|>", special=True, normalized=False),
        ],
    },
    replace_additional_special_tokens=False,
)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

# Safety: if registering the specials actually GREW the vocab, the model's
# embedding matrix is now too small and training/inference would index out of
# range. For Qwen2.5 these tokens already exist, so this is normally a no-op.
_emb_rows = model.get_input_embeddings().weight.shape[0]
if len(tokenizer) > _emb_rows:
    print(f"Vocab grew {_emb_rows} -> {len(tokenizer)}; resizing token embeddings.")
    model.resize_token_embeddings(len(tokenizer))

if getattr(model, "config", None) is not None:
    model.config.eos_token_id = im_end_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.eos_token_id = im_end_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id or im_end_id

# Verify the turn terminator is atomic (single token id) inside a realistic turn.
_sample = "<|im_start|>assistant\nGrace be unto you.<|im_end|>"
_ids = tokenizer(_sample, add_special_tokens=False).input_ids
print("EOS token:", repr(tokenizer.eos_token), "| EOS id:", tokenizer.eos_token_id)
print("<|im_end|> id:", im_end_id, "| decode:", repr(tokenizer.decode([im_end_id])))
print("Sample turn token ids:", _ids)
assert im_end_id in _ids, (
    "<|im_end|> did NOT tokenize atomically -- the training data terminator will "
    "split into ordinary subwords and the model will never learn a clean stop. "
    "Re-tokenize the dataset (Notebook D) with this same fixed tokenizer first."
)
print("OK: <|im_end|> is atomic and set as EOS. Training will teach a clean stop token.")

# Add PEFT adapter matrices
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",

                    "gate_proj", "up_proj", "down_proj"],print("Model and LoRA layers successfully prepared.")

    lora_alpha=32,

    lora_dropout=0,       # Set to 0 to enable Unsloth custom CUDA kernel optimizations)

    bias="none",    random_state=42,
    use_gradient_checkpointing="unsloth",

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

EOS token: '<|im_end|>' | EOS id: 151645


Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model and LoRA layers successfully prepared.


## 3. Load Prepared Datasets

In [4]:
import shutil
from datasets import load_from_disk

# Kaggle read-only mapping fallback: Copy dataset to working directory before mapping tokenization caches
LOCAL_TRAIN_PATH = "/kaggle/working/qa_dataset_train_local" if IS_KAGGLE else TRAIN_DS_PATH
LOCAL_VAL_PATH = "/kaggle/working/qa_dataset_val_local" if IS_KAGGLE else VAL_DS_PATH

if IS_KAGGLE:
    if os.path.exists(LOCAL_TRAIN_PATH):
        shutil.rmtree(LOCAL_TRAIN_PATH)
    shutil.copytree(TRAIN_DS_PATH, LOCAL_TRAIN_PATH)
    if os.path.exists(LOCAL_VAL_PATH):
        shutil.rmtree(LOCAL_VAL_PATH)
    shutil.copytree(VAL_DS_PATH, LOCAL_VAL_PATH)
    print("Copied datasets to local workspace for read-write compatibility.")

train_ds = load_from_disk(LOCAL_TRAIN_PATH)
val_ds = load_from_disk(LOCAL_VAL_PATH)

print(f"Loaded {len(train_ds):,} train examples and {len(val_ds):,} val examples.")

Copied datasets to local workspace for read-write compatibility.
Loaded 2,647 train examples and 140 val examples.


## 4. Training Arguments & SFT Config

In [5]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # Effective batch size = 16
    num_train_epochs=3,                 # 3 epochs is sufficient for SFT
    warmup_steps=50,
    learning_rate=1e-4,                 # Fine-tuning rate (lower than Phase 1 pre-train)
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,                      # IMPORTANT: Do not pack Q&A pairs together to keep distinct boundary contexts
    seed=42,
)

## 5. Initialize Trainer & Apply Serialization Fix

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

# CRITICAL: Train only on the assistant responses (mask system + user tokens).
# This forces the model to put nearly all of its loss signal on the response
# content AND the terminal <|im_end|> stop token, so it actually learns when to
# stop. Without this, the EOS/<|im_end|> signal is too weak and the model never
# stops at inference (degenerating into multilingual garbage after the answer).
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Sanity check: confirm the assistant response is the only part with active labels
# (-100 == masked/ignored) AND that the trained span contains the ATOMIC <|im_end|>
# id. We check the id (not the decoded string) because a corrupted tokenizer can
# decode 151645 to junk like "vinfos" even when the id itself is correct.
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
_sample = trainer.train_dataset[0]
_labels = _sample["labels"]
_unmasked_ids = [tok for tok, lab in zip(_sample["input_ids"], _labels) if lab != -100]
print("Unmasked (trained) token count:", len(_unmasked_ids))
print("Last 8 trained token ids:", _unmasked_ids[-8:])
print("Decoded trained span:\n", tokenizer.decode(_unmasked_ids))
assert im_end_id in _unmasked_ids, (
    f"The atomic <|im_end|> id ({im_end_id}) is NOT in the trained span -- the model "
    "will not learn to stop. The dataset terminator likely split into ordinary "
    "subwords. Re-run the model-load cell (it registers <|im_end|> as atomic) and, "
    "if this still fails, re-tokenize the dataset (Notebook D) with the fixed tokenizer."
)
print(f"OK: atomic <|im_end|> id ({im_end_id}) is in the trained span. Clean stop will be learned.")

# Apply SFTConfig serialization/pickling crash fix (modules mismatch workaround)
try:
    import sys
    if 'trl.trainer.sft_config' in sys.modules:
        sys.modules['trl.trainer.sft_config'].SFTConfig = trainer.args.__class__
        print("Successfully bound SFTConfig serialization hook.")
except Exception as e:
    print(f"Warning binding serialization hook: {e}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2647 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/140 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/2647 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/2647 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/140 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/140 [00:00<?, ? examples/s]

Unmasked (trained) token count: 103
Decoded trained span:
 Hear, therefore, the earnest counsel of the Lord: between the preacher and his soul‑won child there exists a bond of the utmost affection, a father‑son communion that even death cannot sunder. Though the blessed kingdom know not the ties of earthly marriage, this spiritual fatherhood shall endure forever, unbroken by the grave. Thus, as long as the preacher leads his charge to the saving grace of Christ, the love that binds them shall be an everlasting testament to the mercy of God.<|im_end|>

OK: assistant response + <|im_end|> stop token are in the trained span.
Successfully bound SFTConfig serialization hook.


## 6. Execute Training

In [7]:
# Execute training run
trainer_stats = trainer.train()

print(f"Training finished. Time: {trainer_stats.metrics['train_runtime']:.2f}s")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151665}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,647 | Num Epochs = 3 | Total steps = 498
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
200,1.102028,1.276207
400,0.913693,1.276612
498,0.922698,1.281280


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training finished. Time: 4358.12s


## 7. Save PEFT Adapter

In [8]:
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Successfully saved LoRA adapter weights to {FINAL_ADAPTER_DIR}")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/spurgeon_lora_qa/tokenizer_config.json.


Successfully saved LoRA adapter weights to /kaggle/working/spurgeon_lora_qa
